# Kelompok **nanti aja namainnya**

Anggota Kelompok:
1. Bintang Fikri Fauzan (122140008)
2. Ferdana Al Hakim (122140012)
3. Zidan Raihan (122140100)

# **main.ipynb — Notebook untuk menjalankan model klasifikasi**

**Tujuan:** notebook ini dibuat agar panitia bisa menjalankan model yang sudah dibuat (train / eval / inferensi) **dengan menjalankan setiap cell secara berurutan** (Run All). Notebook telah disusun per-cell (Markdown + Code) agar mudah diedit dan dipahami.


**Catatan penting sebelum menjalankan:**
- Pastikan file `model.pth` (atau nama checkpoint Anda) berada di folder yang sama dengan notebook.
- Pastikan `test.csv` dan  folder `test/` tersedia pada direktori kerja saat inferensi.


# Overview  
Model yang digunakan adalah **EfficientNet_B4**, untuk **klasifikasi gambar makanan Indonesia**.  

---

## Spesifikasi Model  
- **Arsitektur**: EfficientNet_B4
- **Jumlah Parameter**: 17.5M  
- **Pre-trained**: ImageNet (IMAGENET1K_V1)  
- **Input Size**: 224 × 224 × 3  
- **Output Classes**: 5 kelas makanan Indonesia  

### Kelas Target  
Model dilatih untuk mengenali 5 jenis makanan Indonesia:  
1. Bakso  
2. Gado-gado  
3. Nasi Goreng  
4. Rendang  
5. Soto Ayam  

---

## Hyperparameters  
- **Optimizer**: Adam  
- **Learning Rate**: 0.0001  
- **Weight Decay**: 0.0001  
- **Batch Size**: 32  
- **Epochs**: 30  
- **Loss Function**: CrossEntropyLoss  
- **Early Stopping**: (Patience=5 & Delta=0.001)

---

## Data Split  
- **Training**: 80% dari dataset training  
- **Validation**: 20% dari dataset training  

---

## Key Features  
- **Transfer Learning**: Menggunakan pre-trained weights dari ImageNet  
- **Fine-tuning**: Classifier layer diganti untuk 5 kelas makanan  
- **Data Augmentation**: Resize, Random flip, rotation, color jitter, dan normalize sesuai ImageNet  
- **Reproducibility**: Fixed Random Seed untuk Python, Numpy dan Pytorch

---

## Keunggulan EfficientNet-B4  
- **Seimbang antara akurasi dan efisiensi**: Dirancang supaya tidak hanya dalam tapi juga lebar dan resolusinya pas, jadi hasil prediksi lebih baik tanpa boros komputasi  
- **Akurasi tinggi**: Lebih unggul dibanding MobileNetV3 untuk dataset gambar yang lebih sulit atau detail  
- **Transfer learning siap pakai**: Sudah dilatih di ImageNet, jadi bisa lebih cepat beradaptasi untuk dataset khusus seperti makanan Indonesia  
- **Ukuran pas**: Dengan ~19 juta parameter, B4 masih cukup ringan tapi punya akurasi lebih tinggi, cocok untuk dataset yang tidak terlalu besar  



# 1) Import, konfigurasi device, dan set seed (reproducibility)


Set seed untuk `random`, `numpy`, dan `torch` serta konfigurasi deterministik yang disarankan pada TOR kompetisi.

In [11]:
import os
import torch
import torch.nn as nn
import timm
import pandas as pd
import torchinfo
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from torchinfo import summary

# device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


# 2) Konfigurasi Path

Mengatur path untuk file dan folder yang akan digunakan selama proses inferensi, serta mendefinisikan kelas-kelas makanan yang menjadi target klasifikasi.

- **DATA_DIR**: Direktori kerja utama (di-set ke folder saat ini).
- **TEST_CSV**: Path ke file `test.csv` yang berisi daftar gambar untuk pengujian.
- **TEST_IMG_DIR**: Path ke folder `test/` yang berisi gambar-gambar uji.
- **MODEL_PATH**: Path ke file model terlatih (`model.pth`) yang akan digunakan untuk inferensi.
- **CLASS_NAMES**: Daftar nama kelas makanan sesuai dengan kompetisi.
- **label2idx**: Dictionary untuk mengubah nama kelas menjadi indeks numerik.
- **idx2label**: Dictionary untuk mengubah indeks numerik kembali ke nama kelas.


In [3]:
# Path folder & file (pakai os)
DATA_DIR = "."
TEST_CSV = os.path.join("test.csv")
TEST_IMG_DIR = os.path.join("test")
MODEL_PATH = os.path.join("model.pth")

# Definisikan Kelas
CLASS_NAMES = ["nasi_goreng", "rendang", "soto_ayam", "bakso", "gado_gado"]
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2label = {i: c for c, i in label2idx.items()}

# 3) Dataset + transform

1. **Transformasi Gambar untuk Inference**
   - Menggunakan `transforms.Compose` dari torchvision untuk membuat pipeline transformasi gambar:
     - `Resize((224, 224))`: Mengubah ukuran gambar menjadi 224x224 piksel (ukuran standar untuk banyak model CNN).
     - `ToTensor()`: Mengubah gambar PIL menjadi tensor PyTorch.
     - `Normalize(mean, std)`: Menormalkan nilai piksel gambar menggunakan mean dan standar deviasi yang umum dipakai pada model ImageNet.

2. **Kelas Dataset Kustom**
   - Mendefinisikan kelas `FoodDataset` (turunan dari `torch.utils.data.Dataset`) untuk memudahkan pemuatan data gambar:
     - `__init__`: Menyimpan dataframe, path folder gambar, dan transformasi.
     - `__len__`: Mengembalikan jumlah data (baris pada dataframe).
     - `__getitem__`: Mengambil satu gambar berdasarkan indeks, membuka file gambar, mengubah ke RGB, menerapkan transformasi, dan mengembalikan tensor gambar beserta nama filenya.

Dengan cell ini, pipeline pemuatan dan transformasi gambar untuk proses inferensi menjadi lebih terstruktur dan siap digunakan pada DataLoader.

In [4]:
# transform untuk inference
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class FoodDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        img_path = os.path.join(self.img_dir, row["filename"])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["filename"]


# 4) Load model

Model yang telah dilatih sebelumnya akan dimuat dari file `model.pth`. Proses pemuatan model dilakukan menggunakan fungsi `torch.load`, dan model akan ditempatkan pada device (CPU atau GPU) yang sudah dikonfigurasi sebelumnya.


Pastikan file `model.pth` sudah tersedia di direktori kerja sebelum menjalankan cell ini.

In [17]:
# Load backbone dengan pretrained
weights = EfficientNet_B4_Weights.IMAGENET1K_V1
model = efficientnet_b4(weights=weights)

# Ganti classifier sesuai jumlah kelas
num_classes = 5
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, num_classes)
)

# Load Model
model.load_state_dict(torch.load("model.pth", map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print("Loaded full model from model.pth")

# Print arsitektur lengkap
print("\n=== Model Architecture ===")
print(summary(model, input_size=(1, 3, 224, 224)))

Loaded full model from model.pth

=== Model Architecture ===
Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [1, 5]                    --
├─Sequential: 1-1                                       [1, 1792, 7, 7]           --
│    └─Conv2dNormActivation: 2-1                        [1, 48, 112, 112]         --
│    │    └─Conv2d: 3-1                                 [1, 48, 112, 112]         1,296
│    │    └─BatchNorm2d: 3-2                            [1, 48, 112, 112]         96
│    │    └─SiLU: 3-3                                   [1, 48, 112, 112]         --
│    └─Sequential: 2-2                                  [1, 24, 112, 112]         --
│    │    └─MBConv: 3-4                                 [1, 24, 112, 112]         2,940
│    │    └─MBConv: 3-5                                 [1, 24, 112, 112]         1,206
│    └─Sequential: 2-3                                  [1, 32, 56, 56]    

# 5) Inference test.csv → jawaban.csv

Pada cell ini dilakukan proses inferensi pada data uji dan penyimpanan hasil prediksi:

- **Membaca test.csv**  
  DataFrame `df_test` berisi daftar nama file gambar yang akan diuji.

- **Dataset & DataLoader**  
  Membuat objek `FoodDataset` dan `DataLoader` untuk memudahkan pemrosesan batch gambar selama inferensi.

- **Proses Inferensi**  
  Untuk setiap batch gambar:
  - Gambar dipindahkan ke device (CPU/GPU).
  - Model melakukan prediksi.
  - Hasil prediksi diambil indeks kelas tertinggi (`argmax`).
  - Indeks diubah ke label kelas menggunakan `idx2label`.
  - Nama file dan label prediksi disimpan.

- **Simpan jawaban.csv**  
  Hasil prediksi disimpan ke file `jawaban.csv` dengan dua kolom: `filename` dan `label`.

In [6]:
# Baca test.csv
df_test = pd.read_csv(TEST_CSV)
print("Jumlah data test:", len(df_test))

# Dataset & DataLoader
test_ds = FoodDataset(df_test, TEST_IMG_DIR, transform=test_transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Inference dengan tqdm
preds, fnames = [], []
with torch.no_grad():
    for imgs, names in tqdm(test_loader, desc="Inferencing", unit="batch"):
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        pred_idx = outputs.argmax(dim=1).cpu().numpy().tolist()
        preds.extend([idx2label[p] for p in pred_idx])
        fnames.extend(names)

# Simpan jawaban.csv
out_df = pd.DataFrame({"filename": fnames, "label": preds})
out_df.to_csv("jawaban.csv", index=False)
print("Selesai! File jawaban.csv dibuat dengan", len(out_df), "baris.")

Jumlah data test: 100


Inferencing: 100%|██████████| 4/4 [00:04<00:00,  1.02s/batch]

Selesai! File jawaban.csv dibuat dengan 100 baris.
